In [1]:
import yfinance as yf
import pandas as pd

In [2]:
dat = yf.Ticker("NKE")   

# Daily prices (adjusted), then take last trading day of each month
d = dat.history(period="17y", interval="1d", auto_adjust=True)

s = d["Close"].tz_localize(None).resample("M").last()   # Series indexed by month-end
s.index.name = "month_end"

# Make month_end a normal column and add an integer index column
monthly = s.reset_index(name="adj_close")
monthly.insert(0, "idx", range(1, len(monthly)+1))   # 1-based; use range(len(monthly)) for 0-based

#Change column names
monthly=monthly.rename(columns={"month_end":"Date", "adj_close":"Price"})


#Drop idx column
monthly=monthly.drop(columns=["idx"])

#Convert to DataFrame
microsoft_price = pd.DataFrame(monthly)

# Add column called calendar_quarter

p = microsoft_price["Date"].dt.to_period("Q")        # calendar quarters
microsoft_price["cal_year"] = p.dt.year.astype("Int64")    # 2010, 2011, ...
microsoft_price["cal_q"]    = p.dt.quarter                 # 1..4

# Final label: "Q{cal_q} {cal_year}"
microsoft_price["calendar_quarter"] = "Q" + microsoft_price["cal_q"].astype(str) + " " + microsoft_price["cal_year"].astype(str)


#Drop cal_year and cal_q column
microsoft_price=microsoft_price.drop(columns=["cal_year", "cal_q"])


microsoft_price


/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_11888/16179352.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  s = d["Close"].tz_localize(None).resample("M").last()   # Series indexed by month-end


,Date,Price,calendar_quarter
0,2008-09-30,13.345490,Q3 2008
1,2008-10-31,11.496272,Q4 2008
2,2008-11-30,10.622531,Q4 2008
3,2008-12-31,10.222830,Q4 2008
4,2009-01-31,9.070256,Q1 2009
...,...,...,...
200,2025-05-31,59.878819,Q2 2025
201,2025-06-30,70.672722,Q2 2025
202,2025-07-31,74.303856,Q3 2025
203,2025-08-31,76.970001,Q3 2025


In [3]:
microsoft_price.to_csv("nike_price.csv")